# Food Label Detective Agent — Exploration & Walkthrough

This notebook explores the dataset and demonstrates each part of the agent:
data lookup (Open Food Facts + CSV fallback), the analysis engine, guardrails,
and the full agent workflow. It runs fully offline using the synthetic CSV.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src import off_client, preprocessing, guardrails, agent
from src import preferences as prefs_module

## 1. Explore the synthetic starter dataset

In [ ]:
df = pd.read_csv('../data/sample_products.csv')
print(df.shape)
df[['product_name', 'sugars_100g', 'salt_100g', 'nova_group', 'nutriscore_grade']]

## 2. Fetch a product (API with CSV fallback)

In [ ]:
product = off_client.resolve_product('Nutella')
product['product_name'], product['source'], product['ingredients_text']

## 3. Run the deterministic analysis engine

In [ ]:
analysis = preprocessing.build_label_analysis(product)
print('Allergens :', analysis['allergens'])
print('Additives :', analysis['additives'])
print('Dietary   :', analysis['dietary'])
print('Risk      :', analysis['risk']['score'], analysis['risk']['band'])
analysis['risk']['reasons']

## 4. Apply guardrails against a user's preferences

In [ ]:
prefs = {'diet': 'vegan', 'low_sugar': True, 'low_salt': False, 'allergies': ['milk']}
guard = guardrails.apply_guardrails(product, analysis, prefs)
print('Recommendation:', guard['recommendation'])
for c in guard['conflicts']:
    print(f"  [{c['severity']}] {c['message']}")

## 5. Full agent workflow (end to end)

In [ ]:
prefs_module.save_preferences('nb_user', prefs)
result = agent.analyse_product('Nutella', user_id='nb_user')
print('Status:', result['status'])
print('Explanation source:', result['explanation_source'])
print()
print(result['explanation'])

## 6. Compare risk scores across the dataset

In [ ]:
rows = []
for _, r in df.iterrows():
    p = off_client._row_to_product(r.to_dict())
    risk = preprocessing.compute_risk_score(p)
    rows.append({'product': p['product_name'], 'score': risk['score'], 'band': risk['band']})
pd.DataFrame(rows).sort_values('score', ascending=False)